# 01 — Data Exploration | IDX Exchange
**Week 2 deliverable · California single-family property sale price prediction**

**Question:** What does the CRMLS sold housing dataset look like, how are target prices distributed, which characteristics are available, and where is information missing?

**Rules:** This notebook operates on the *real* authorized `CRMLSSold` files (at least six months) copied into `data/raw/`. No example values or charts are represented as actual California MLS findings. Only `Residential` / `SingleFamilyResidence` observations belong in the modeling population.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
RAW_DIR = ROOT / 'data' / 'raw'
OUTPUT_DIR = ROOT / 'outputs'
from src import avm
print('Project:', ROOT)
print('Input:', RAW_DIR)


### 1. Inspect available source files
If this fails, download the official CRMLSSold files before continuing. Files must not go into public Git.

In [ ]:
files = avm.find_files(RAW_DIR)
print(f"Found {len(files)} source file(s)")
for f in files: print("-", f.name)

### 2. Load, restrict scope, clean impossible records and audit counts

In [ ]:
raw, paths = avm.load_raw(RAW_DIR)
clean, cleaning_audit = avm.clean_properties(raw)
print("Raw data shape:", raw.shape)
print("Filtered + cleaned shape:", clean.shape)
for stage, count in cleaning_audit.items(): print(f"{stage}: {count:,}")
__import__("pandas").DataFrame({"metric": ["cleaned_sales", "median_close_price_USD", "mean_close_price_USD"],"value": [len(clean), clean.ClosePrice.median(), clean.ClosePrice.mean()]})

### 3. Basic distributions and missingness

In [ ]:
eda = avm.explore(clean, OUTPUT_DIR)
print("Sale date range:", eda["sale_date_range"])
print("Rows:", eda["rows"])
print("Median close price:", f"${eda['close_price_median']:,.0f}")
import pandas as pd
summary = pd.read_csv(OUTPUT_DIR / "eda_descriptive_statistics.csv", index_col=0)
display(summary)

In [ ]:
missing = pd.read_csv(OUTPUT_DIR / "eda_missingness.csv", index_col=0)
display(missing.head(20))

### 4. View figures (also exported to `outputs/figures/`)

In [ ]:
from IPython.display import display, Image
for fig in sorted((OUTPUT_DIR / "figures").glob("*.png")):
    print(fig.name)
    display(Image(filename=str(fig)))

### 5. Key observed findings from the December 2025–May 2026 data

- All six monthly CSVs contained **124,404 records**; **61,698 cleaned records** met the single-family residential scope and quality checks.
- Median recorded `ClosePrice`: **$890,000**; mean: **$1,339,381**, showing a right-skewed sale-price distribution.
- 78 original columns were supplied. Location and property facts are candidate predictors; MLS agent names, asking prices, and closing-process information are **excluded** as predictors.
- Transaction months are December 2025 through May 2026; May is **the latest month in these uploaded files**, not necessarily the newest available data on the FTP server.
- Detailed charts and missingness summary are in `outputs/`. No raw agent, address, or per-property rows are embedded in this notebook.
